In [18]:
from langgraph.checkpoint.memory import MemorySaver  # Para memoria a nivel de hilo
from langgraph.store.memory import InMemoryStore  # Para memoria a largo plazo (usuario)
from langgraph.graph.message import AnyMessage, add_messages  # Para gestionar mensajes en el estado del grafo
from langgraph.managed.is_last_step import RemainingSteps  # Para rastrear el límite de recursión

# Inicializar la memoria a largo plazo
in_memory_store = InMemoryStore()

# Inicializar el checkpointer para memoria a nivel de hilo
checkpointer = MemorySaver()


In [19]:
from dotenv import load_dotenv
import openai
import os
from langchain_core.tools import tool
from extract_to_cmaps import prompt_extract_cmapss, generate_cmapss_assistant_prompt
import json
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
# Carga las variables de entorno desde el archivo .env
load_dotenv(dotenv_path=".env", override=True)

# Recupera la clave API de OpenAI desde las variables de entorno
openai.api_key = os.getenv("OPENAI_API_KEY")

# Verifica que la clave API se haya cargado correctamente
if openai.api_key is None:
    raise ValueError("La clave API de OpenAI no está configurada correctamente.")

# Ahora puedes usar la clave API en LangChain o directamente con OpenAI
llm = ChatOpenAI(temperature=0, model="gpt-3.5-turbo")  # O el modelo que prefieras


In [20]:
from typing_extensions import TypedDict
from typing import Annotated, List

class State(TypedDict):
    user_id: str
    messages: Annotated[list[AnyMessage], add_messages]
    
    # Historial de mensajes
    messages: Annotated[list[AnyMessage], add_messages]
    # loaded_memory: Stores information loaded from the long-term memory store, 
    # typically user preferences or historical context.
    loaded_memory: str
    # remaining_steps: Used by LangGraph to track the number of allowed steps 
    # to prevent infinite loops in cyclic graphs.
    remaining_steps: RemainingSteps 


In [ ]:

from langchain_core.tools import tool # Decorator to define a function as a LangChain tool
import ast # Module to safely evaluate strings containing Python literal structures
from extract_to_cmaps import prompt_extract_cmapss, generate_cmapss_assistant_prompt

# @tool
# def extract_cmapss_data(message: str):
#     """
#     Esta herramienta extrae los datos estructurados para alimentar un modelo de predicción RUL basado en CMAPSS.
#     """
#     print("\nRecibiendo mensaje:", message)  # Verifica que el mensaje llega
#     prompt = prompt_extract_cmapss(message)
#     print("\nPrompt generado:", prompt)  # Verifica que el prompt se genera correctamente
    
#     # Llamada al modelo
#     response = llm(prompt)
#     print("\nRespuesta del modelo:", response)  # Verifica lo que devuelve el modelo
    
#     try:
#         # Intentamos parsear la respuesta para asegurar que está en formato JSON
#         parsed_response = json.loads(response)
#     except json.JSONDecodeError:
#         # Si la respuesta no es un JSON válido, devolver un mensaje de error
#         parsed_response = {
#             "error": "La respuesta del modelo no es un JSON válido",
#             "modelo_seleccionado": "FD001"  # Selección por defecto
#         }
#     return json.dumps(parsed_response)

@tool
def extract_cmapss_data(message: str):
    """
    Extraer datos del estado de un motor aeronáutico para alimentar un modelo de predicción RUL basado en CMAPSS.
    """
    print(f"Extrayendo datos del mensaje: {message}")
    
    # Simulamos la extracción de los datos
    extracted_data = {
        "unidad": 200,
        "tiempo_ciclos": 150,
        "configuraciones_operativas": [1, 2, 3],
        "mediciones_sensores": {
            "s_1": 0.25, "s_2": 0.35, "s_3": 0.45, "s_7": 550
        },
        "modelo_seleccionado": "FD001"
    }
    
    return extracted_data



In [5]:
from langgraph.prebuilt import ToolNode  # Nodo predefinido para ejecutar herramientas
from langchain_core.runnables import RunnableConfig

# Nodo para el asistente que maneja la extracción de datos
def cmapss_assistant(state: State, config: RunnableConfig): 
    print("Ejecutando cmapss_assistant...")  # Indica que estamos dentro del nodo de asistente
    
    # Recibir la memoria cargada
    memory = state["loaded_memory"] if "loaded_memory" in state else "None"
    
    # Aquí generamos el prompt para el asistente basado en la memoria cargada.
    cmapss_assistant_prompt = f"""
    Eres un asistente especializado en la extracción de datos para alimentar modelos de predicción RUL basados en CMAPSS.
    Contexto de la memoria del usuario: {memory}
    Mensaje del usuario: {state["messages"][-1].content}  # Último mensaje del usuario
    """
    
    print(f"Prompt generado para el asistente: {cmapss_assistant_prompt}")  # Ver el prompt generado

    # Este paso simplemente devuelve el mensaje generado por el asistente
    return {"messages": [cmapss_assistant_prompt]}


In [6]:
from langgraph.prebuilt import ToolNode

from langgraph.graph import StateGraph, START, END 

# Nodo de herramienta que ejecuta la extracción de datos
cmapss_tool_node = ToolNode([extract_cmapss_data])

# Crear el grafo de estados
cmapss_workflow = StateGraph(State)

# Agregar nodos al grafo
cmapss_workflow.add_node("cmapss_assistant", cmapss_assistant)  # Nodo de asistente
cmapss_workflow.add_node("cmapss_tool_node", cmapss_tool_node)  # Nodo de herramienta

# Conectar los nodos en el flujo
cmapss_workflow.add_edge(START, "cmapss_assistant")

In [7]:
# Modificación de should_continue para siempre continuar con la herramienta.
def should_continue(state: State, config: RunnableConfig):
    print("Ejecutando should_continue...")  # Indica que estamos evaluando la condición
    last_message = state["messages"][-1]
    
    # Forzar la continuación, ya que no detectamos un tool_call explícito
    print("Forzamos la ejecución de la herramienta, continuamos.")  # Siempre continuamos a la herramienta
    return "continue"  # Fuerza que el flujo pase a cmapss_tool_node

In [8]:
# Usar la versión ajustada de should_continue para que siempre pasemos a cmapss_tool_node
cmapss_workflow.add_conditional_edges(
    "cmapss_assistant",  # Nodo fuente
    should_continue,  # Función de condición ajustada
    {
        "continue": "cmapss_tool_node",  # Siempre ejecutamos cmapss_tool_node
        "end": END,  # No lo utilizamos, ya que siempre seguimos
    }
)

# Volver al asistente después de ejecutar la herramienta
cmapss_workflow.add_edge("cmapss_tool_node", "cmapss_assistant")

# Compilar el agente
cmapss_agent = cmapss_workflow.compile(name="cmapss_agent", checkpointer=checkpointer, store=in_memory_store)


In [9]:
# Ahora ejecutamos el agente con el mensaje del usuario
import uuid

# Crear un ID único para la sesión
thread_id = uuid.uuid4()

# Mensaje de prueba
question = "Datos del motor con ID 200, ciclo operativo 150, configuraciones 1, 2, 3."

# Configuración para el agente
config = {"configurable": {"thread_id": thread_id}}

# Ejecutar el agente con el mensaje
print(f"Ejecutando el agente con el mensaje: {question}")  # Mensaje antes de la ejecución
result = cmapss_agent.invoke({"messages": [HumanMessage(content=question)]}, config=config)

# Mostrar los mensajes en el estado final
print("Estado final después de ejecutar el agente:")  # Ver el estado final
for message in result["messages"]:
    message.pretty_print()  # Imprimir los mensajes de salida

Ejecutando el agente con el mensaje: Datos del motor con ID 200, ciclo operativo 150, configuraciones 1, 2, 3.
Ejecutando cmapss_assistant...
Prompt generado para el asistente: 
    Eres un asistente especializado en la extracción de datos para alimentar modelos de predicción RUL basados en CMAPSS.
    Contexto de la memoria del usuario: None
    Mensaje del usuario: Datos del motor con ID 200, ciclo operativo 150, configuraciones 1, 2, 3.  # Último mensaje del usuario
    
Ejecutando should_continue...
Forzamos la ejecución de la herramienta, continuamos.


ValueError: No AIMessage found in input

In [13]:
def run_agent_with_message(question, config):
    print(f"Ejecutando el agente con el mensaje: {question}")
    
    # Generar el mensaje humano
    human_message = HumanMessage(content=question)
    
    # Configuración adicional (si es necesaria)
    print("Generando el prompt y configurando la ejecución...")

    # Ejecución del agente
    try:
        # Ejecutar el agente
        result = cmapss_agent.invoke({"messages": [human_message]}, config=config)
        
        # Depuración: Mostrar todos los mensajes en el resultado
        print("Mensajes generados en la ejecución del agente:")
        for msg in result.get("messages", []):
            print(f"Tipo de mensaje: {type(msg).__name__}, Contenido: {msg.content}")
        
        # Verificar si el resultado contiene un AIMessage
        ai_message_found = any(isinstance(msg, AIMessage) for msg in result.get("messages", []))
        
        if ai_message_found:
            print("AIMessage encontrado. Continuamos con el flujo.")
        else:
            print("No se encontró un AIMessage en el resultado.")
        
        # Verificar el resultado final
        print("Estado final después de ejecutar el agente:")
        print(result)
        
    except Exception as e:
        print(f"Error en la ejecución del agente: {e}")
        # Aquí puedes agregar más detalles de manejo de errores si es necesario

# Llamar a la función para probar
run_agent_with_message("Datos del motor con ID 200, ciclo operativo 150, configuraciones 1, 2, 3.", config)


Ejecutando el agente con el mensaje: Datos del motor con ID 200, ciclo operativo 150, configuraciones 1, 2, 3.
Generando el prompt y configurando la ejecución...
Ejecutando cmapss_assistant...
Prompt generado para el asistente: 
    Eres un asistente especializado en la extracción de datos para alimentar modelos de predicción RUL basados en CMAPSS.
    Contexto de la memoria del usuario: None
    Mensaje del usuario: Datos del motor con ID 200, ciclo operativo 150, configuraciones 1, 2, 3.  # Último mensaje del usuario
    
Ejecutando should_continue...
Forzamos la ejecución de la herramienta, continuamos.
Error en la ejecución del agente: No AIMessage found in input
